# 非正規GLMによる異常検知

ここまで扱ってきたGLMは、確率分布に正規分布、リンク関数に恒等関数を用いた線形回帰モデルでした。
statsmodelsライブラリを用いると、それ以外の分布やリンク関数を選んでGLMを構築することも可能です。ここでは一例として、確率分布にガンマ分布、リンク関数に対数関数を用いた、以下の式で表されるガンマ回帰モデルを紹介します。

<img src="https://latex.codecogs.com/svg.image?
\begin{align}
&\eta=w_1x+w_0 \\
&\log \theta = \eta \\
&p(y \mid x,w_1,w_0,k) = Ga(\theta,k)
\end{align}
" />

このリッジ回帰モデルによる異常検知を、Pythonを用いて以下の手順で実装する方法を解説します。

- A. モデルの学習
- B. 推論

なおデータセットには7.2節で作成したサンプルデータを使用し、入力する変数は`year`（経過年数）を、ターゲットとする誤報率としては0.0027を採用します（1変数線形回帰モデルと同様）。

## A. モデルの学習

ガンマ回帰モデルの最尤推定による学習と、分位点に基づく異常度のしきい値を算出します。

statsmodelsで確率分布やリンク関数をカスタマイズしながらGLMを実装するには、[statsmodels.genmod.generalized_linear_model.GLM](https://www.statsmodels.org/stable/generated/statsmodels.genmod.generalized_linear_model.GLM.html)クラスを使用し、`family`引数に確率分布やリンク関数を指定するクラスを渡します。確率分布は[statsmodels.genmod.families.family](https://www.statsmodels.org/stable/glm.html#families)から、リンク関数は[statsmodels.genmod.families.links](https://www.statsmodels.org/stable/glm.html#link-functions)から該当するクラスを選択します。

In [ ]:
# コード7.15 ガンマ回帰による1 説明変数の異常検知の実装例（学習）
import pandas as pd
import numpy as np
import statsmodels.api as sm

###### 学習データの読み込みと前処理（1変数線形回帰モデルと同様）######
# CSVからPandas DataFrameにデータ読み込み
df = pd.read_csv('./datasets/usedcar_dataset_train.csv')
# 正常データのみを抽出
df_normal = df[df['label'] == 'normal']
# 学習データの説明変数（'year'）と応答変数（'price'）を
# 別々に保持（応答変数のみndarray化）
x_train = df_normal['year']
y_train = df_normal['price'].to_numpy()

###### 学習ステップ1. 正常のモデルを作成する######
X_train_intercept = sm.add_constant(x_train) # 切片を追加
family=sm.families.Gamma(sm.families.links.Log()) # 確率分布とリンク関数を指定
mod = sm.GLM(y_train, X_train_intercept, family=family) # GLMモデルを作成
res = mod.fit() # モデルの学習を実行

###### 学習ステップ2. 異常を表す指標（異常度）を定義する######
# 式を定義するのみでプログラム上は処理を実施しない

###### 学習ステップ3. 異常度にしきい値を設ける######
TARGET_FP_RATE = 0.0027 # ターゲットとする誤報率（正規分布の3σ相当=0.0027）
# 学習データの説明変数から確率分布を計算
gen = res.get_distribution(exog=X_train_intercept)
# 上記確率分布と学習データの応答変数から確率密度を求める
train_pd = gen.pdf(y_train)
train_anom_score = -np.log(train_pd) # 異常度を求める
# 異常度の分位点からしきい値算出
a_th = np.quantile(train_anom_score, 1-TARGET_FP_RATE)

###### 学習で求めたしきい値を表示######
print(f'a_th={a_th}')

学習済みモデル`res`の`get_distribution`メソッドを呼び出すと、推定済みパラメータを反映した確率分布オブジェクト（4章で解説した`scipy.stats.gamma`形式）が得られます。
推定されたパラメータから求めた確率密度関数と学習データを重ねてプロットしてみます。

In [ ]:
# 学習したガンマ回帰モデルの確率密度関数と学習データを重ねてプロット
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

###### 確率密度関数を描画 ######
# (x,y)格子点を作成
x_min, x_max = np.min(x_train), np.max(x_train)
y_min, y_max = np.min(y_train), np.max(y_train)
x_grid = np.arange(x_min-1, x_max+2)
y_grid = np.linspace(y_min-50, y_max+50, num=200)
X, Y = np.meshgrid(x_grid, y_grid)
XY_grid = np.c_[X.ravel(), Y.ravel()]
# 確率密度関数
X_grid_intercept = sm.add_constant(XY_grid[:, 0])
gen_grid = res.get_distribution(exog=X_grid_intercept)
grid_pd = gen_grid.pdf(XY_grid[:, 1])
# 確率密度をプロット
X_grid_pivot = grid_pd.reshape(X.shape)  # ピボット化
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(5, 5))
ax.contourf(X, Y, X_grid_pivot, levels=10, cmap=cm.gray, alpha=0.5)

###### 学習データを散布図で描画 ######
sns.scatterplot(x=x_train, y=y_train,
                c='#333333', ax=ax, s=18, marker="o")
ax.set_xlabel('year')
ax.set_ylabel('price')
# グラフを表示
plt.show()

線形回帰モデルと比べると、リンク関数に対数関数を使用しているため、説明変数`year`（横軸）に対する応答変数`price`（縦軸）の分布の変化が線形になっていないことがわかります。線形回帰モデルではリンク関数に恒等関数を用いていたため、説明変数と応答変数の線形な関係しか表現できませんでした。一方でガンマ回帰モデルではリンク関数に対数関数を用いることで、非線形な関係も表現できます。

モデルの保持方法は線形回帰の場合と同様、学習で求めたパラメータを数値で保存することもできますが、選択した確率分布や変数によって保存内容が変わるため、実装が煩雑となりがちです。代わりに学習済みモデルのインスタンス`res`をpickle形式で保存するのが実用的です。

In [ ]:
# コード7.16 pickle によるstatsmodels のGLM モデルの保存
import pickle
filepath = './7_6_1_trained_glm_model.pkl' # モデルの保存ファイル名
with open(filepath,'wb') as p: #ファイルを開く
    pickle.dump(res, p) # pickleでファイルにモデルを保存する

## B. 推論

学習フェーズで保存したモデルとしきい値を用いて、推論データに対する異常度の算出と異常判定を行います。

In [ ]:
# コード7.17 ガンマ回帰による1 説明変数の異常検知の実装例（推論）
###### 学習済みモデルとしきい値を読み込み######
filepath = './7_6_1_trained_glm_model.pkl' # モデルの保存ファイル名
with open(filepath,'rb') as p: #ファイルを開く
    res = pickle.load(p) # pickle形式ファイルからモデルを読み込み
A_TH=7.452036657153986 # 異常度のしきい値

###### 推論データの読み込みと前処理（1変数線形回帰モデルと同様）######
# CSVからPandas DataFrameにデータ読み込み
df_inference = pd.read_csv('./datasets/usedcar_dataset_inference.csv')
# 推論データの説明変数（'year'）と応答変数（'price'）をNumpyのndarray化
x_inference = df_inference['year'].to_numpy()
y_inference = df_inference['price'].to_numpy()

###### 推論を実行######
# 異常度を算出
X_inference_intercept = sm.add_constant(x_inference) # 切片を追加
# 学習データの説明変数から確率分布を計算
gen = res.get_distribution(exog=X_inference_intercept)
# 上記確率分布と学習データの応答変数から確率密度を求める
train_pd = gen.pdf(y_inference)
anomaly_scores = -np.log(train_pd) # 異常度を求める
# しきい値により異常の有無を判定
pred = np.where(anomaly_scores > A_TH, 'anomaly', 'normal')
# 推論結果を表示
print(pred)

推論結果の決定境界（異常と正常の判定の境界）を可視化してみます。

In [ ]:
# 推論結果の決定境界を可視化
# 描画用のFigureとAxesを生成
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(5, 5))

###### 正常と異常の範囲を色分け ######
# (x,y)格子点を作成
x_min, x_max = np.min(x_inference), np.max(x_inference)
y_min, y_max = np.min(y_inference), np.max(y_inference)
x_grid = np.arange(x_min-1, x_max+2)
y_grid = np.linspace(y_min-50, y_max+50, num=500)
X, Y = np.meshgrid(x_grid, y_grid)
XY_grid = np.c_[X.ravel(), Y.ravel()]
# 異常度を算出
X_grid_intercept = sm.add_constant(XY_grid[:, 0])
gen_grid = res.get_distribution(exog=X_grid_intercept)
grid_pd = gen_grid.pdf(XY_grid[:, 1])  # 上記確率分布と学習データの応答変数から確率密度を求める
anomaly_scores_grid = -np.log(grid_pd)  # 異常度を求める
# しきい値判定
pred_grid = np.where(anomaly_scores_grid > A_TH, 0, 1)
# 正常と異常の境界をプロット
pred_pivot = pred_grid.reshape(X.shape)
ax.contourf(X, Y, pred_pivot, levels=1,
            cmap=cm.gray, alpha=0.5)

###### 各データを散布図としてプロット ######
sns.scatterplot(data=df_inference, x='year', y='price',
                hue='label', palette=['#999999', '#111111'], ax=ax)
# 凡例を追加
ax.legend()
# グラフを表示
plt.show()

推定した確率密度関数に応じて、決定境界が引かれていることがわかります。